In [1]:
import sys
import os
import importlib
import pandas as pd

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s › %(message)s",
    datefmt="%H:%M:%S",
)


# a raiz do projeto (onde fica o notebook) deve estar no path
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# Célula 2 – recarregar módulos alterados
# ──────────────────────────────────────────────────────────────────────────────
import importlib

import extract.sheets_fetcher   as sf_mod
import treat.utils.write_back   as wb_mod      # agora separado para evitar circular import
import treat.treat_pipeline     as tp_mod
import treat.treat_runner       as tr_mod

importlib.reload(sf_mod)
importlib.reload(wb_mod)
importlib.reload(tp_mod)
importlib.reload(tr_mod)


<module 'treat.treat_runner' from '/home/debrito/Documentos/etl_debrito/treat/treat_runner.py'>

In [ ]:
creds_path     = "creds.json"
spreadsheet_id = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
sheet_name     = "tiktokAlcance"

In [4]:
from extract.sheets_fetcher import SheetsFetcher

fetcher = SheetsFetcher(spreadsheet_id, creds_path)
df_dict = fetcher.get([sheet_name])
df_raw  = df_dict[sheet_name]

print(f"▶️ Dados crus: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 15)
df_raw.head(3)

13:06:24 INFO extract.sheets_fetcher › 📡 batchGet 1 ranges


▶️ Dados crus: 665 linhas × 9 colunas


,date,account_name,campaign_name,ad_group_name,ad_name,objective,utm_content,reach,imrpessions
0,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_CARD_CINTIA_ACAO_DBT_SBRAE_2025_EMP_...,CONSIDERATION,dbt_sbrae_2025_emp_fem0113,60758,0
1,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_CARD_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM...,CONSIDERATION,dbt_sbrae_2025_emp_fem0114,81520,0
2,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_CARD_SILVANA_ACAO_DBT_SBRAE_2025_EMP...,CONSIDERATION,dbt_sbrae_2025_emp_fem0115,18781,0


In [5]:
from treat.treat_pipeline        import TreatPipeline
from treat.utils.renomeacoes     import renomeacao_geral

pipeline = TreatPipeline(
    creds_path         = creds_path,
    spreadsheet_id     = spreadsheet_id,
    sheet_name         = sheet_name,
    mapping_renomeacao = renomeacao_geral,
    write_back         = False,    # mudar para True em produção
)

df_ok = pipeline.run(df_raw)

# mostrar todas as colunas, mas só 5 linhas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
df_ok.head(10)

13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'impressions' ausente em df_raw ou df_ok; pulando aggregate check
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'account_name' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'campaign_name' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'ad_group_name' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'ad_name' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'objective' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'utm_content' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'reach' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'imrpessions' vazia em 1 linha(s): 664
13:06:29 WARNING treat.utils.validations › [Validação] Coluna 'start' v

,date,account_name,campaign_name,ad_group_name,ad_name,objective,utm_content,reach,imrpessions,start,end,Campanha,ID_Campanha,Veiculo,ID_Veiculo,Compartilhamentos,Comentarios,Reacoes,Engajamento_Total,ID
0,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_CARD_CINTIA_ACAO_DBT_SBRAE_2025_EMP_...,Tráfego,dbt_sbrae_2025_emp_fem0113,60758,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
1,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_CARD_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM...,Tráfego,dbt_sbrae_2025_emp_fem0114,81520,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
2,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_CARD_SILVANA_ACAO_DBT_SBRAE_2025_EMP...,Tráfego,dbt_sbrae_2025_emp_fem0115,18781,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
3,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_...,Tráfego,dbt_sbrae_2025_emp_fem0110,127346,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
4,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_...,Tráfego,dbt_sbrae_2025_emp_fem0111,48696,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
5,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPC_BRASIL, 18+, POPULAÇÃO EM GE...",2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_...,Tráfego,dbt_sbrae_2025_emp_fem0111,58970,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
6,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPM_BRASIL, 18+, POPULAÇÃO EM GE...",carrossel_pinterest,Alcance,dbt_sbrae_2025_emp_fem0112,98928,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
7,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPM_BRASIL, 18+, POPULAÇÃO EM GE...",video_horizontal_manifesto_15,Alcance,dbt_sbrae_2025_emp_fem0111,53379,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
8,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPM_BRASIL, 18+, POPULAÇÃO EM GE...",video_horizontal_manifesto_30,Alcance,dbt_sbrae_2025_emp_fem0110,145623,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---
9,2025-03-08,DeBrito - SEBRAE,2025_3_EMPREENDEDORISMO FEMININO_ALC_COMERCIAL...,"2025_3_BR_ALC_CPM_BRASIL, 18+, POPULAÇÃO EM GE...",video_horizontal_manifesto_6,Alcance,dbt_sbrae_2025_emp_fem0111,61468,0,2025-03-08,2025-03-31,Empreendedorismo Feminino,dbt_sbrae_2025_emp_fem,Pinterest,164,0,0,0,0,2025-03-08-Empreendedorismo Feminino---


In [6]:
import importlib
import treat.bi_param_utils as bp
importlib.reload(bp)

import importlib, utils.preview_links as pl
importlib.reload(pl)


df_ok = pipeline.run(df_raw)
import importlib, treat.treat_pipeline as tp
importlib.reload(tp)



13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'impressions' ausente em df_raw ou df_ok; pulando aggregate check
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'account_name' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'campaign_name' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'ad_group_name' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'ad_name' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'objective' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'utm_content' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'reach' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'imrpessions' vazia em 1 linha(s): 664
13:06:31 WARNING treat.utils.validations › [Validação] Coluna 'start' v

<module 'treat.treat_pipeline' from '/home/debrito/Documentos/etl_debrito/treat/treat_pipeline.py'>

In [7]:
# ── DEBUG DO FALLBACK POR campaign_name ────────────────────────────────────
from treat.bi_param_utils import BIParamLookup
import pandas as pd
from IPython.display import display

# 1) já temos df_ok do pipeline; vamos filtrar só as linhas com utm_content vazio
df_fallback = df_ok[df_ok["utm_content"].astype(str).str.strip() == ""]

print("Linhas com utm_content vazio:", df_fallback.shape[0])
if df_fallback.empty:
    print("Nenhuma linha nesse cenário, nada a testar.")
else:
    # 2) instanciar lookup e BI_PARAM
    lookup = BIParamLookup(creds_path, spreadsheet_id)
    lookup._ensure_df()
    df_param = lookup._df

    # descobrir coluna de taxonomy_campaign_name
    cols = {c.strip().lower(): c for c in df_param.columns}
    taxonomy_col = cols.get("taxonomy_campaign_name")

    # 3) séries de teste
    df_fallback = df_fallback.copy()
    df_fallback["__key_fallback"] = (
        df_fallback["campaign_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # 4) mapa de taxonomy_campaign_name → campaign_name/utm_campaign no BI
    raw = lookup._map_columns(
        key_kw="taxonomy_campaign_name",
        val_kws=["campaign_name", "utm_campaign"],
        upper_keys=False
    )
    fb_camp_map = {k.strip().lower(): v[0] for k, v in raw.items()}
    fb_utm_map  = {k.strip().lower(): v[1] for k, v in raw.items()}

    # 5) aplica mapeamento
    df_fallback["__mapped_Campanha_fb"]   = df_fallback["__key_fallback"].map(fb_camp_map)
    df_fallback["__mapped_ID_Campanha_fb"]= df_fallback["__key_fallback"].map(fb_utm_map)

    # 6) mostra resultados
    display(
      df_fallback[[
        "campaign_name",
        "__key_fallback",
        "__mapped_Campanha_fb",
        "__mapped_ID_Campanha_fb"
      ]].head(20)
    )

    faltantes_fb = df_fallback["__mapped_Campanha_fb"].isna().sum()
    print(f"\nFallback: {faltantes_fb} de {len(df_fallback)} linhas sem correspondência em taxonomy_campaign_name")


Linhas com utm_content vazio: 1


,campaign_name,__key_fallback,__mapped_Campanha_fb,__mapped_ID_Campanha_fb
664,,,NaN,NaN



Fallback: 1 de 1 linhas sem correspondência em taxonomy_campaign_name
